# RouteKV Compiler — Phase 1 Baseline Profiler

This notebook:
1. Simulates a decode trace with synthetic KV pages
2. Runs LRU baseline vs route-aware policy
3. Compares estimated stall time across tiers
4. Plots per-step stall and tier utilisation

Run on Colab with: `pip install git+https://github.com/harsha-mangena/routekv-compiler.git`

In [ ]:
!pip install -q git+https://github.com/harsha-mangena/routekv-compiler.git

In [ ]:
import random, math
from routekv.compiler.ir import KVPage
from routekv.compiler.planner import TieringPlanner, TierBudgetRatio
from routekv.runtime.simulator import KVPageSimulator
from routekv.policies.baselines import recency_policy
from routekv.policies.route_aware import route_aware_policy, score_gap

random.seed(42)
N_PAGES = 128
N_STEPS = 256

def synthetic_pages(n, context_type='retrieval'):
    pages = []
    for i in range(n):
        # retrieval: high route_score variance; chat: high hotness
        if context_type == 'retrieval':
            rs = random.betavariate(2, 1)
            ht = random.betavariate(1, 3)
        else:
            rs = random.betavariate(1, 3)
            ht = random.betavariate(2, 1)
        pages.append(KVPage(
            layer=i % 32, head_group=0,
            start_token=i*128, end_token=(i+1)*128,
            bytes_estimate=1024*64,
            hotness=round(ht, 3),
            route_score=round(rs, 3),
        ))
    return pages


In [ ]:
planner = TieringPlanner(budget=TierBudgetRatio(hbm=0.25, dram=0.45, storage=0.30))

results = {}
for context in ['retrieval', 'chat']:
    pages = synthetic_pages(N_PAGES, context)
    # LRU baseline
    lru_pages = recency_policy(pages)
    lru_decisions = planner.plan(lru_pages)
    lru_sim = KVPageSimulator(pages=lru_pages, decisions=lru_decisions, n_steps=N_STEPS)
    lru_res = lru_sim.run()
    # Route-aware
    ra_pages = route_aware_policy(pages)
    ra_decisions = planner.plan(ra_pages)
    ra_sim = KVPageSimulator(pages=ra_pages, decisions=ra_decisions, n_steps=N_STEPS)
    ra_res = ra_sim.run()
    results[context] = {'lru': lru_sim.summary(lru_res), 'route_aware': ra_sim.summary(ra_res)}
    print(f'\n=== {context} ===')
    print(f"  LRU          avg_stall={results[context]['lru']['avg_stall_us']:.2f} µs")
    print(f"  Route-aware  avg_stall={results[context]['route_aware']['avg_stall_us']:.2f} µs")
    print(f"  Score gap: {score_gap(pages):.2f} ranks")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, context in zip(axes, ['retrieval', 'chat']):
    pages = synthetic_pages(N_PAGES, context)
    lru_dec = planner.plan(recency_policy(pages))
    ra_dec = planner.plan(route_aware_policy(pages))
    lru_stalls = [r.estimated_stall_us for r in KVPageSimulator(pages, lru_dec, N_STEPS).run()]
    ra_stalls = [r.estimated_stall_us for r in KVPageSimulator(pages, ra_dec, N_STEPS).run()]
    ax.plot(lru_stalls, label='LRU baseline', alpha=0.7)
    ax.plot(ra_stalls, label='RouteKV (route-aware)', alpha=0.7)
    ax.set_title(f'{context} workload')
    ax.set_xlabel('Decode step')
    ax.set_ylabel('Estimated stall (µs)')
    ax.legend()
plt.suptitle('RouteKV: LRU vs Route-Aware KV Tiering')
plt.tight_layout()
plt.savefig('routekv_phase1_baseline.png', dpi=150)
plt.show()
